## Pistes de détection d'anomalies


Mise en oeuvre de règles de contrôles sur les valeurs présentes dans le jeu de données 

### L’index de fin est inférieur à l’index de début

In [16]:
import pandas as pd
import matplotlib.pyplot as plt

df_contrats = pd.read_csv("../data/contrats.csv")
df_factures = pd.read_csv("../data/factures.csv", parse_dates=["date_facture"])
etiquettes = df_factures.pop("anomalie") 
df = pd.merge(df_factures, df_contrats, left_on="identifiant_contrat",
              right_on="identifiant", how="left", validate="many_to_one")
ABONNEMENTS = {3: 11.50, 6: 15.60, 9: 19.50, 12: 23.30}#liste de clés et valeurs pour les abonnements selon la puissance souscrite


df["regle_index"] = df["index_fin"]<df["index_debut"]
print("Nombre de factures avec index_debut > index _ fin : ",df["regle_index"].sum())


df["abonnement"] = df["puissance_kva"].map(ABONNEMENTS)
df["montant_attendu"] = ((df["consommation_kwh"]*df["prix_kwh"])+df["abonnement"])

#print (df["montant_attendu"])
#Comparaison regle montant

df["regle_montant"] = (df["montant"]-df["montant_attendu"]).abs()>0.01

print("Nombre de factures avec montant <> au recalcul (>0.01€) : ",df["regle_montant"].sum())
factures_ecart = df[df["regle_montant"]]
#print (factures_ecart)
df.loc[df["regle_montant"],["id_facture", "identifiant_contrat", "date_facture","montant_attendu","montant"]]



Nombre de factures avec index_debut > index _ fin :  8
Nombre de factures avec montant <> au recalcul (>0.01€) :  9


,id_facture,identifiant_contrat,date_facture,montant_attendu,montant
98,F00099,C0009,2025-03-01,104.6310,941.67
131,F00132,C0011,2025-12-01,57.6370,172.92
230,F00231,C0020,2025-03-01,73.1358,438.84
290,F00291,C0025,2025-03-01,145.2367,1742.88
316,F00317,C0028,2025-02-01,171.5519,2058.60
339,F00340,C0030,2025-01-01,225.2688,2703.24
494,F00495,C0043,2025-11-01,81.3552,488.16
520,F00521,C0046,2025-09-01,77.3159,695.88
522,F00523,C0046,2025-11-01,122.1772,1099.62


In [17]:
df["mediane_contrat"] = df.groupby("identifiant_contrat")["consommation_kwh"].transform("median")

df["ratio"]=df["consommation_kwh"]/df["mediane_contrat"]
df["regle_conso"] = df["ratio"].abs()>3

print("Nombre de factures avec la consommation <> comparée à son contrat : ",df["regle_conso"].sum())
df.loc[df["regle_conso"], ["id_facture", "identifiant_contrat", "date_facture", "consommation_kwh", "mediane_contrat", "ratio"]]

Nombre de factures avec la consommation <> comparée à son contrat :  10


,id_facture,identifiant_contrat,date_facture,consommation_kwh,mediane_contrat,ratio
51,F00052,C0005,2025-04-01,1240,130.0,9.538462
61,F00062,C0006,2025-02-01,6420,428.5,14.982497
127,F00128,C0011,2025-08-01,740,131.5,5.627376
171,F00172,C0015,2025-04-01,1100,124.5,8.835341
193,F00194,C0017,2025-02-01,5190,350.0,14.828571
210,F00211,C0018,2025-07-01,2060,366.0,5.628415
348,F00349,C0030,2025-10-01,4110,522.5,7.866029
384,F00385,C0034,2025-09-01,2670,389.5,6.854942
486,F00487,C0043,2025-03-01,2940,267.0,11.011236
489,F00490,C0043,2025-06-01,1620,267.0,6.067416


In [18]:
df["alerte"] = df["regle_index"] | df["regle_montant"] | df["regle_conso"]
print("Factures en alerte :", df["alerte"].sum())

Factures en alerte : 27


In [19]:
pd.crosstab(df["regle_index"], etiquettes.fillna("normale"), margins=True, margins_name="Total",rownames=["Règle index"], colnames=["Vérité"])

Vérité,consommation_aberrante,index_recule,montant_errone,normale,Total
Règle index,,,,,
False,10,0,9,545,564
True,0,8,0,0,8
Total,10,8,9,545,572


In [20]:
pd.crosstab(df["regle_montant"], etiquettes.fillna("normale"), margins=True, margins_name="Total",rownames=["Règle montant"], colnames=["Vérité"])

Vérité,consommation_aberrante,index_recule,montant_errone,normale,Total
Règle montant,,,,,
False,10,8,0,545,563
True,0,0,9,0,9
Total,10,8,9,545,572


In [21]:
pd.crosstab(df["regle_conso"], etiquettes.fillna("normale"), margins=True, margins_name="Total",rownames=["Règle conso"], colnames=["Vérité"])

Vérité,consommation_aberrante,index_recule,montant_errone,normale,Total
Règle conso,,,,,
False,0,8,9,545,562
True,10,0,0,0,10
Total,10,8,9,545,572


In [22]:
pd.crosstab(df["alerte"], etiquettes.fillna("normale"), margins=True, margins_name="Total")

anomalie,consommation_aberrante,index_recule,montant_errone,normale,Total
alerte,,,,,
False,0,0,0,545,545
True,10,8,9,0,27
Total,10,8,9,545,572


### Tableau de synthèse des performances des règles

| Règle | Anomalie visée | Signalées | Correctes  | Précision  | Rappel  |
| :--- | :--- | :---: | :---: | :---: | :---: |
| **Index** | index_recule | 8 | 8 | 100% | 100% |
| **Montant** | montant_errone | 9 | 9 | 100% | 100% |
| **Consommation** | consommation_aberrante | 10 | 10 | 100% | 100% |
| **Alerte (les 3)** | toutes | 27 | 27 | 100% | 100% |

*Note : Pour la ligne "Alerte (les 3)", le nombre total d'anomalies (dénominateur du rappel) correspond à la somme de toutes les anomalies uniques*

La tolérance de la règle 2 (règle montant): 
Le problème a deux causes 
le générateur et le notebook n’arrondissaient pas les demi-centimes de la même façon (24,865 € donnait 24,87 d’un côté et 24,86 de l’autre) - un écart de « 0,01 » se calcule en réalité en 0.010000000000001563, qui dépasse 0,01.

La correction a d’ailleurs consisté à ne plus arrondir le montant attendu cf les factures F00008, F00036, F00382 vues précédemment 
(Montant :24,87	/ Attendu 24,86)

La règle 2 - règle montant - face aux consommations aberrantes () : Le montant est recalculé à partir d’une consommation déjà fausse : il est donc cohérent avec elle. La règle 2 vérifie le calcul, pas la donnée d’entrée.

Le choix du seuil de 3 pour la règle 3 (règle conso):
• Ratio maximal d'une facture normale :( 1.5 * 1.1 ) / 1 = 1.65


• Ratio minimal d'une facture (x10) :
Une consommation initialement basse (mois faible à 0,6) mais multipliée par 10 donne un coefficient de 6. Ramené à la médiane du contrat (1), le ratio est de :

(0.6*10)/ 1 = 6

Conclusion : Le seuil de 3 tombe parfaitement entre les deux valeurs (1,65 < 3 < 6,0). Il est assez haut pour éviter les fluctuations normales (jusqu'à 1,65) et assez bas pour capturer une anomalie flagrante de type (* 10) (qui démarre à 6,0).

Les cas où la règle 3 - règle conso - échouerait :
• Historique trop court (Contrat résilié à 2 factures) : Si un contrat ne compte que deux factures dont l'une est normale (ex: 100 kWh) et l'autre est \(\times 10\) (ex: 1000 kWh), la médiane statistique sera la moyenne des deux (550 kWh). Le ratio de la facture anormale ne sera que de \(1000 / 550 = 1,81\). Ce score étant inférieur au seuil de 3, l'anomalie passe totalement inaperçue.
• Anomalies discrètes (\(\times 1,5\)) : Si une consommation frauduleuse ou un dysfonctionnement n'augmente le volume que de 1,5, le ratio obtenu se fondra complètement dans le bruit des variations saisonnières normales (qui atteignent un ratio maximal de 1,65). La règle échouera à la détourer du signal normal.

Les fausses alertes possibles:
• Saisonnalité extrême non corrigée : Un client équipé d'un chauffage électrique climatisation réversible peut consommer très peu en intersaison mais subir un pic massif lors d'un hiver polaire ou d'une canicule, franchissant naturellement le seuil de 3 par rapport à sa médiane annuelle.
• Contrats courts à forte rupture saisonnière : Un contrat activé en plein été (faible consommation) qui ne comporte que 2 factures estivales verra sa médiane s'établir très bas. Dès que la première facture d'hiver (chauffage) arrivera, le saut de consommation sera perçu à tort comme une anomalie par la règle.

**Conclusion**

Chaque règle déterministe cible un angle précis du problème : la règle 1 valide la chronologie des index, la règle 2 verrouille la cohérence du calcul financier, et la règle 3 isole les volumes de consommation disproportionnés par rapport à l'historique du client. Grâce à des critères de simulation nets, le système atteint ici un score parfait de 100 % de détection. Cependant, cette performance exceptionnelle est artificielle, car nos données synthétiques intègrent des anomalies franches, sans aucun bruit de mesure ni comportements clients atypiques. L'enjeu de l'étape 4 sera donc de confronter cette approche à la réalité du terrain. Plutôt que de remplacer nos règles actuelles ou de faire disparaître les arbitrages de seuils, l'introduction de modèles statistiques permettra de changer de perspective : la question sera de mesurer si une approche agnostique, qui ne dispose d'aucune règle métier préétablie, est capable de détecter ces mêmes anomalies de manière autonome, et d'évaluer son efficacité face au risque de fausses alertes.